## ReLU MLP Single-Seed Learning Dynamics

This notebook runs a standard fully connected ReLU network with a single training phase.

It focuses on one seed to inspect learning dynamics clearly (checkpoints, scatter alignment, and start/end metrics).

No dropout, normalization, or multi-phase logic is used.

In [ ]:
from pathlib import Path

from itables import show
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from itertools import product

%load_ext autoreload
%autoreload 2

from layer_lenses.odt import samples_reaching_node
from layer_lenses.odt import odt_leaf_ids_for_x
from layer_lenses.relu_analysis import (
    collect_start_end_metrics,
    first_layer_odt_alignment,
    masked_accuracy_by_odt_leaf,
    neuron_leaf_responsibility_from_ablation,
    plot_first_layer_odt_alignment,
    plot_layered_relu_graph,
    quadratic_snapshot_epochs,
    run_single_relu_seed,
    summarize_neuron_responsibility,
)
from layer_lenses.relu_analysis import log_loss_gradients
from layer_lenses.relu_training import checkpoint_model_from_state
from layer_lenses.relu_analysis import activation_tensor_by_layer_neuron_data

from pathlib import Path
import pickle
from typing import Any

def save_seed_result_pickle(seed_result: dict[str, Any], path: str | Path) -> Path:
    """Save seed_result to a pickle file."""
    out_path = Path(path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("wb") as f:
        pickle.dump(seed_result, f, protocol=pickle.HIGHEST_PROTOCOL)
    return out_path


def load_seed_result_pickle(path: str | Path) -> dict[str, Any]:
    """Load seed_result from a pickle file."""
    in_path = Path(path)
    with in_path.open("rb") as f:
        obj = pickle.load(f)
    if not isinstance(obj, dict):
        raise TypeError(f"Expected dict from pickle, got {type(obj).__name__}")
    return obj


In [ ]:


# Scratch-aligned defaults with a plain ReLU MLP.
DIM = 100
DEPTH = 4
N_TRAIN = 1_000_000
HIDDEN_DIMS = [30,30,30]

TOTAL_EPOCHS = 10_000
LR = 20e-3
BATCH_SIZE = 10_000
WEIGHT_DECAY = 0.01
MOMENTUM = 0.999
SNAPSHOT_EPOCHS = list[int](range(0,200)) + list[int](range(200,TOTAL_EPOCHS,100))
print(f"Using {len(SNAPSHOT_EPOCHS)} snapshot epochs.")

SEED = 5178

seed_result = None

## Run selected seed

This is the expensive cell. It runs single-phase ReLU MLP training for the selected seed.

In [ ]:
print(f"Running seed {SEED} ...")
seed_result = run_single_relu_seed(
    SEED,
    dim=DIM,
    depth=DEPTH,
    n_train=N_TRAIN,
    hidden_dims=HIDDEN_DIMS,
    total_epochs=TOTAL_EPOCHS,
    lr=LR,
    momentum=MOMENTUM,
    batch_size=BATCH_SIZE,
    weight_decay=WEIGHT_DECAY,
    snapshot_epochs=SNAPSHOT_EPOCHS,
    project_first_layer_grad_orthogonal=None,
    odt_hyperplanes = "random_orthogonal",
    bias= False,
    optimizer="sgd",
    device="cuda"
)
print("Done")

In [ ]:


# Save
# save_path = save_seed_result_pickle(seed_result, "results/cache/seed_result_5178_5_40_40_40_1Mtrain_10kbatch_adam_randomhyper_fixed_order_data.pkl")
# print("Saved to:", save_path)

# Load
seed_result = load_seed_result_pickle("results/cache/seed_result_replayable_5_40_40_40_1Mtrain_10kbatch_sgd_momentum_9e-1.pkl")
print(seed_result.keys()) 

In [ ]:
seed_result["data"]["x_train"].shape

In [ ]:
plt.plot(seed_result["out"]["epoch_losses"][0:])
out = seed_result["out"]
ckpts = out["checkpoint_snapshots"]
tree = seed_result["tree"]

## Neuron-wise First layer neuron alignment analysis

In [ ]:
for name, param in ckpts[9900].items():
    print(name, param.shape)

In [ ]:
FIRST_LAYER_NEURON_ID = [0]
ODT_INTERNAL_NODE_ID = [0,1,2]
START_EPOCH_PLOT_IDX = 0
END_EPOCH_PLOT_IDX = 20



neuron_ids = list(FIRST_LAYER_NEURON_ID)
odt_node_ids = list(ODT_INTERNAL_NODE_ID)

if not neuron_ids:
    raise ValueError("FIRST_LAYER_NEURON_ID must not be empty.")
if not odt_node_ids:
    raise ValueError("ODT_INTERNAL_NODE_ID must not be empty.")

first_layer_width = ckpts[0]["hidden_layers.0.weight"].shape[0]
internal_node_count = tree.w_list.shape[0]

for neuron_id in neuron_ids:
    if not (0 <= neuron_id < first_layer_width):
        raise ValueError(
            f"Each FIRST_LAYER_NEURON_ID must be in [0, {first_layer_width - 1}], "
            f"got {neuron_id}."
        )

for odt_node_id in odt_node_ids:
    if not (0 <= odt_node_id < internal_node_count):
        raise ValueError(
            f"Each ODT_INTERNAL_NODE_ID must be in [0, {internal_node_count - 1}], "
            f"got {odt_node_id}."
        )

epochs = sorted(ckpts.keys())[START_EPOCH_PLOT_IDX:END_EPOCH_PLOT_IDX]

odt_vectors = {
    odt_node_id: torch.from_numpy(tree.w_list[odt_node_id]).float()
    for odt_node_id in odt_node_ids
}
odt_norms = {
    odt_node_id: float(torch.norm(vector).clamp_min(1e-12))
    for odt_node_id, vector in odt_vectors.items()
}

fig, axes = plt.subplots(
    nrows=len(neuron_ids),
    ncols=1,
    figsize=(8, 4.5 * len(neuron_ids)),
    sharex=True,
    squeeze=False,
)
axes = axes[:, 0]

for ax, neuron_id in zip(axes, neuron_ids):
    neuron_norms = []
    output_wts = []
    normalized_inner_products = {
        odt_node_id: [] for odt_node_id in odt_node_ids
    }

    for epoch in epochs:
        w = (
            ckpts[epoch]["hidden_layers.0.weight"][neuron_id]
            .detach()
            .cpu()
            .float()
        )
        w_norm = float(torch.norm(w).clamp_min(1e-12))
        neuron_norms.append(w_norm)
        output_wts.append(ckpts[epoch]["output_layer.weight"][0,neuron_id].detach().cpu().float())
        for odt_node_id in odt_node_ids:
            cosine = float(
                torch.dot(w, odt_vectors[odt_node_id])
                / (w_norm * odt_norms[odt_node_id])
            )
            normalized_inner_products[odt_node_id].append(cosine)

    for odt_node_id in odt_node_ids:
        label = (
            "normalized inner product (cosine)"
            if len(odt_node_ids) == 1
            else f"ODT node {odt_node_id} cosine"
        )
        ax.plot(
            epochs,
            normalized_inner_products[odt_node_id],
            linewidth=2.0,
            label=label,
        )

    ax.plot(
        epochs,
        neuron_norms,
        linewidth=2.0,
        label="neuron norm",
    )
    ax.plot(
        epochs,
        output_wts,
        linewidth=2.0,
        label="output weight",
    )
    ax.axhline(1.0, color="gray", linestyle="--", linewidth=1)
    ax.axhline(0.0, color="gray", linestyle="--", linewidth=1)
    ax.axhline(-1.0, color="gray", linestyle="--", linewidth=1)
    ax.set_ylabel("Value")
    # ax.set_yscale("log")
    ax.set_title(
        "First-layer neuron vs ODT node over checkpoints\n"
        f"neuron={neuron_id}, odt_nodes={odt_node_ids}"
    )
    ax.set_ylim(-2,10)
    ax.grid(alpha=0.25)
    ax.legend()

axes[-1].set_xlabel("Checkpoint epoch")
fig.tight_layout()
plt.show()

## Scatter plot  

In [ ]:
EPOCH = list(ckpts.keys())[297]  # choose from available checkpoints

alignment, fig, ax = plot_first_layer_odt_alignment(
    state_dict=ckpts[EPOCH],
    tree=tree,
    depth=DEPTH,
    epoch=EPOCH,
    seed=SEED,
)
plt.show()

# Keep these names available for downstream scratch cells.
first_layer_wts = alignment["first_layer_wts"]
odt_wts = alignment["odt_wts"]
row_norms = alignment["row_norms"]
cos = alignment["cos"]
abs_cos = alignment["abs_cos"]
max_abs_cos = alignment["max_abs_cos"]
closest_odt = alignment["closest_odt"]
closest_odt_np = alignment["closest_odt_np"]
node_level = alignment["node_level"]

state = ckpts[EPOCH]
HIDDEN_WEIGHTS = [
    state[f"hidden_layers.{layer_idx}.weight"].detach().cpu()
    for layer_idx in range(len(HIDDEN_DIMS))
]
W0 = HIDDEN_WEIGHTS[0]
Wout = state["output_layer.weight"].detach().cpu()



## neuron merge analysis

In [ ]:
W = ckpts[EPOCH]["hidden_layers.0.weight"]   # (150, 100)
v = ckpts[EPOCH]["output_layer.weight"][0]   # (150,)
u0 = torch.as_tensor(tree.w_list[0], dtype=W.dtype)
u0 = u0 / u0.norm()

w = W[[69, 114]]
p = v[[69, 114]] * torch.linalg.vector_norm(w, dim=1)  # signed products
align = (w @ u0) 
orth = torch.linalg.vector_norm(w - align[:, None] * u0, dim=1)
cos_pair = torch.nn.functional.cosine_similarity(w[0], w[1], dim=0)

In [ ]:
def merge_pair(state, i=69, j=114, ODT_wts):
    s = {k: t.clone() for k, t in state.items()}
    w, v = s["hidden_layers.0.weight"], s["output_layer.weight"] 
    # the two w vectors will be positive scalar multiples of each other. The v scalars are opposite in sign. In the defauult example w_i and w_j are along u_0
    
    # collapse j into i in function space, then zero j
    p_i = w[i] / w[i].norm()
    p_j = w[j]/w[j].norm()          # same gate ⇒ products add
    p = p_i + p_j
    p = p / p.norm()
    # simpler: put the net map on i, kill j
    v[0, i] = v[0, i]*w[i].norm() + v[0, j]*w[j].norm()
    w[i] = p
    
    w[j] = 0
    v[0, j] = 0
    return s

In [ ]:
merged_ckpt = merge_pair(ckpts[EPOCH], ODT_wts=odt_wts)
print(merged_ckpt["hidden_layers.0.weight"][69].norm())
print(merged_ckpt["hidden_layers.0.weight"][114].norm())
print(merged_ckpt["output_layer.weight"][0,69])
print(merged_ckpt["output_layer.weight"][0,114])

print(ckpts[EPOCH]["hidden_layers.0.weight"][69].norm())
print(ckpts[EPOCH]["hidden_layers.0.weight"][114].norm())
print(ckpts[EPOCH]["output_layer.weight"][0,69])
print(ckpts[EPOCH]["output_layer.weight"][0,114])



In [ ]:
from layer_lenses.relu_training import evaluate_relu_mlp
model1 = checkpoint_model_from_state(seed_result["out"]["model"], ckpts[EPOCH])
model2 = checkpoint_model_from_state(seed_result["out"]["model"], merged_ckpt)

results_pre_merge = evaluate_relu_mlp(model=model1, x_eval=seed_result["data"]["x_train"], y_eval=seed_result["data"]["y_train"], device="cuda")
results_post_merge = evaluate_relu_mlp(model=model2, x_eval=seed_result["data"]["x_train"], y_eval=seed_result["data"]["y_train"], device="cuda")

print(results_pre_merge["log_loss"])
print(results_post_merge["log_loss"])



## Load hidden-layer and output parameters from a given epoch

In [ ]:
components = torch.diag(row_norms) @ cos
max_w, argmax_w = torch.max(cos.abs(), dim=0)
for input_idx in range(components.shape[1]):
    neuron_idx = int(argmax_w[input_idx])
    print(
        input_idx,
        max_w[input_idx],
        argmax_w[input_idx],
        components[neuron_idx].norm(),
    )


## Visualise neural net as graph

In [ ]:
# Layered ReLU graph at a chosen snapshot epoch.
ALIGNMENT_ABS_THRESHOLD = 0.8
EDGE_ABS_THRESHOLD = 0.01  # tune this; increase if graph is too dense
FIGSIZE = (14, 18)

# One mask per hidden layer; edit individual entries to select nodes.
NODE_MASK = [np.ones(width, dtype=bool) for width in HIDDEN_DIMS]

graph_alignment, visible_edges, graph_fig, graph_ax = plot_layered_relu_graph(
    state_dict=state,
    tree=tree,
    depth=DEPTH,
    epoch=EPOCH,
    node_mask=NODE_MASK,
    alignment_abs_threshold=ALIGNMENT_ABS_THRESHOLD,
    edge_abs_threshold=EDGE_ABS_THRESHOLD,
    figsize=FIGSIZE,
    require_both_edge_endpoints_selected=True,
)
plt.show()

In [ ]:
odt_node = 0
large_alignment_ids = torch.where((components[:,odt_node]/components.norm(dim=1)).abs()>0.8)[0]

for neuron_id in large_alignment_ids:
    print(neuron_id,"\t",components[neuron_id,odt_node],"\t",components[neuron_id].norm(),"\t",Wout[0,neuron_id])


## Computing neuron responsibilities -- i.e. log loss changes to data entering leaf nodes on neuron removal

In [ ]:
checkpoint_model = checkpoint_model_from_state(
    seed_result["out"]["model"],
    ckpts[EPOCH],
)
base_node_mask = [np.ones(width, dtype=bool) for width in HIDDEN_DIMS]


In [ ]:

neuron_responsibility, neuron_responsibility_df, baseline_leaf_metrics = (
    neuron_leaf_responsibility_from_ablation(
        model=checkpoint_model,
        x_eval=seed_result["data"]["x_train"],
        y_eval=seed_result["data"]["y_train"],
        tree=seed_result["tree"],
        base_node_mask=base_node_mask,
        min_log_loss_delta=-2.,
        min_relative_log_loss_delta=None,
        min_accuracy_drop=None,
    )
)
neuron_summary = summarize_neuron_responsibility(neuron_responsibility)


In [ ]:
baseline_leaf_metrics

In [ ]:
layer_select = 0
neuron_select = 30

show(
    neuron_responsibility_df[(neuron_responsibility_df["layer"]==layer_select) & (neuron_responsibility_df["neuron_id"]==neuron_select) ],
    scrollY="500px",
    scrollX=True,
    search=True,
    paging=True,
    pageLength=32,
)

In [ ]:
checkpoint_model = checkpoint_model_from_state(
    seed_result["out"]["model"],
    ckpts[EPOCH],
)

# Keep only selected neurons in one layer while retaining every other layer.
MASKED_LAYER_IDX = 0
MASKED_NEURON_IDS = 94
node_mask = [np.ones(width, dtype=bool) for width in HIDDEN_DIMS]
node_mask[MASKED_LAYER_IDX][:] = False
node_mask[MASKED_LAYER_IDX][MASKED_NEURON_IDS] = True

leaf_accuracy = masked_accuracy_by_odt_leaf(
    model=checkpoint_model,
    node_mask=node_mask,
    x_eval=seed_result["data"]["x_eval"],
    y_eval=seed_result["data"]["y_eval"],
    tree=seed_result["tree"],
)

display(leaf_accuracy)
print(
    "Overall masked accuracy:",
    np.average(
        leaf_accuracy["accuracy"],
        weights=leaf_accuracy["num_samples"],
    ),
)

## Neuron activity breakdown -- neuron activity based on data entering ODT leaf nodes 

In [ ]:

checkpoint_model = checkpoint_model_from_state(seed_result["out"]["model"], ckpts[EPOCH])

acts = activation_tensor_by_layer_neuron_data(
    model=checkpoint_model,
    x_eval=seed_result["data"]["x_train"],
)
outputs = checkpoint_model(torch.tensor(seed_result["data"]["x_train"], dtype=torch.float32)).detach().cpu().numpy()
labels = seed_result["data"]["y_train"]

residues = np.exp(-labels*outputs)/(1+np.exp(-labels*outputs))
leaf_ids = odt_leaf_ids_for_x(seed_result["data"]["x_train"], seed_result["tree"])

print("activation_tensor shape:", acts.shape)  # (layers, neurons, num_data)
print("Residues shape:", residues.shape)


In [ ]:
# from itertools import product
# layer0 = range(HIDDEN_DIMS[0])
# layer1 = range(HIDDEN_DIMS[1])
# layer2 = range(HIDDEN_DIMS[2])
# path_stats = dict()
# for path in product(layer0,layer1,layer2):
#     print(path)
#     acts_path = acts[0,path[0]]*acts[1,path[1]]*acts[2,path[2]]
#     weight_path = (HIDDEN_WEIGHTS[1][path[1],path[0]]*HIDDEN_WEIGHTS[2][path[2],path[1]]*Wout[0,path[2]]).numpy()*np.linalg.norm(W0[path[0]].numpy())
#     acts_path_hist = np.array([np.sum(acts_path[leaf_ids==j]) for j in range(31,63)])
#     first_layer_gating_vec = W0[path[0]].numpy()/np.linalg.norm(W0[path[0]].numpy())
#     path_stats[path] = dict(
#         weight=weight_path,
#         acts_hist=acts_path_hist,
#         first_layer_gating_vec=first_layer_gating_vec,
#     )

# for path in path_stats:
#     path_stats[path]["num_points"] = np.sum(path_stats[path]["acts_hist"])


In [ ]:
n0, n1, n2 = (int(w) for w in HIDDEN_DIMS[:3])
leaf_values = np.arange(31, 63)
n_leaves = leaf_values.size
n_data = acts.shape[-1]

A0 = np.ascontiguousarray(acts[0, :n0], dtype=np.float32)
A1 = np.ascontiguousarray(acts[1, :n1], dtype=np.float32)
A2 = np.ascontiguousarray(acts[2, :n2], dtype=np.float32)
leaf_ids_arr = np.asarray(leaf_ids)

W1 = HIDDEN_WEIGHTS[1][:n1, :n0].detach().cpu().numpy()
W2 = HIDDEN_WEIGHTS[2][:n2, :n1].detach().cpu().numpy()
wout = Wout[0, :n2].detach().cpu().numpy()
W0_np = W0[:n0].detach().cpu().numpy()
w0_norm = np.linalg.norm(W0_np, axis=1)
gating = W0_np / w0_norm[:, None]

# weight[i, j, k] = W1[j, i] * W2[k, j] * Wout[k] * ||W0[i]||
weights = np.einsum("ji,kj,k,i->ijk", W1, W2, wout, w0_norm)

# hist[i, j, k, leaf] = number of points in that leaf on path (i, j, k)
hist = np.zeros((n0, n1, n2, n_leaves), dtype=np.float32)
for leaf_pos, leaf in enumerate(leaf_values):
    idx = np.flatnonzero(leaf_ids_arr == leaf)
    if idx.size == 0:
        continue
    a0 = A0[:, idx]
    a1 = A1[:, idx]
    a2 = A2[:, idx]
    # (n0, n1, n2): for each k, (A0 * A2[k]) @ A1.T
    for k in range(n2):
        hist[:, :, k, leaf_pos] = (a0 * a2[k]) @ a1.T

hist_i64 = np.rint(hist).astype(np.int64)
num_points = hist_i64.sum(axis=3)

path_stats = {
    (i, j, k): dict(
        weight=weights[i, j, k],
        acts_hist=hist_i64[i, j, k],
        first_layer_gating_vec=gating[i],
        num_points=int(num_points[i, j, k]),
    )
    for i in range(n0)
    for j in range(n1)
    for k in range(n2)
}

In [ ]:
strong_paths = [path for path in path_stats if abs(path_stats[path]["weight"])>0.1 and path_stats[path]["num_points"]>1000 and path[0]==0]

In [ ]:
print(HIDDEN_WEIGHTS[2][:,31])
print(Wout)
print((HIDDEN_WEIGHTS[2][:,31]*Wout).sign())



In [ ]:
for path in strong_paths:
    print(path,"\t",path_stats[path]["num_points"],"\t",path_stats[path]["weight"],
                "\n",path_stats[path]["acts_hist"])

In [ ]:


i0=None
i1= 19
i2= None

layer0_selected = np.bool(A0[i0]) if i0 is not None else np.ones(N_TRAIN, dtype=bool)
layer1_selected = np.bool(A1[i1]) if i1 is not None else np.ones(N_TRAIN, dtype=bool)
layer2_selected = np.bool(A2[i2]) if i2 is not None else np.ones(N_TRAIN, dtype=bool)

plt.hist(leaf_ids[layer0_selected*layer1_selected*layer2_selected], bins=np.arange(31,64))

In [ ]:
layer0 = range(HIDDEN_DIMS[0])
layer1 = range(HIDDEN_DIMS[1])
layer2 = range(HIDDEN_DIMS[2])

for path in product([0],layer1,layer2):
    if path not in strong_paths:
        continue
    print(path,"\t",path_stats[path]["num_points"],"\t",path_stats[path]["weight"],
                "\t",path_stats[path]["acts_hist"][-3])


In [ ]:
np.linalg.norm(path_stats[path]["first_layer_gating_vec"])

In [ ]:
ACTIVITY_LAYER_IDX = 0
ACTIVITY_NEURON_ID =30
internal_node_count = tree.w_list.shape[0]
leaf_bins = np.arange(internal_node_count, 2 * internal_node_count + 2)

plt.hist(
    leaf_ids[acts[ACTIVITY_LAYER_IDX, ACTIVITY_NEURON_ID]],
    bins=leaf_bins,
);


residue_weighted_activations = acts[ACTIVITY_LAYER_IDX, ACTIVITY_NEURON_ID] * residues
residue_weighted_activations_leafwise = []

for j in sorted(set(leaf_ids)):
    residue_weighted_activations_leafwise.append(residue_weighted_activations[leaf_ids==j].sum())
plt.figure()
plt.bar(sorted(set(leaf_ids)),residue_weighted_activations_leafwise)

odt_node_focus = 0
residue_weighted_activations_odt_node_focus_component = residue_weighted_activations * (seed_result["data"]["x_train"]@tree.w_list[odt_node_focus])
residue_weighted_activations_odt_node_focus_component_leafwise = []
for j in sorted(set(leaf_ids)):
    residue_weighted_activations_odt_node_focus_component_leafwise.append(residue_weighted_activations_odt_node_focus_component[leaf_ids==j].sum())
plt.figure()
plt.bar(sorted(set(leaf_ids)),residue_weighted_activations_odt_node_focus_component_leafwise)


In [ ]:
from layer_lenses.relu_analysis import first_layer_neuron_weighted_lens

FIRST_LAYER_NEURON_ID = 30
state = ckpts[EPOCH]
checkpoint_model = checkpoint_model_from_state(
    seed_result["out"]["model"],
    state,
)
acts = activation_tensor_by_layer_neuron_data(
    model=checkpoint_model,
    
    x_eval=seed_result["data"]["x_train"],
)

# Sum every active path from this first-layer neuron to the output.
total_lens = first_layer_neuron_weighted_lens(
    state_dict=state,
    activation_tensor=acts,
    first_layer_neuron_id=FIRST_LAYER_NEURON_ID,
)
total_lens *= residues * -labels
grad_alt = (
    total_lens @ seed_result["data"]["x_train"] / len(residues)
)

# This should match the log-loss gradient for the selected first-layer neuron.
grad_check = log_loss_gradients(
    model=checkpoint_model,
    x=seed_result["data"]["x_train"],
    y=seed_result["data"]["y_train"],
)
true_grad = grad_check["hidden_layers.0.weight"][
    FIRST_LAYER_NEURON_ID
].numpy()


In [ ]:
true_grad@grad_alt/(np.linalg.norm(true_grad)*np.linalg.norm(grad_alt))

In [ ]:
for j in range(0,31):
    print(j,true_grad@odt_wts[j].numpy()/(np.linalg.norm(true_grad)*np.linalg.norm(odt_wts[j])))

In [ ]:
for j in range(31,63):
    print(j,total_lens[leaf_ids==j].sum())


In [ ]:
from tqdm.std import tqdm

# Decompose the first-layer neuron's lens through one downstream hidden layer.
# A one-hidden-layer model has no downstream hidden layer, so its selected
# first-layer neuron is the sole component.
FIRST_LAYER_NEURON_ID = 30
PATH_LAYER_IDX = 1 if len(HIDDEN_DIMS) > 1 else 0
PATH_NODE_IDS = (
    range(HIDDEN_DIMS[PATH_LAYER_IDX])
    if PATH_LAYER_IDX > 0
    else [FIRST_LAYER_NEURON_ID]
)
NUM_GRAD_EPOCHS = 100
analysis_epochs = sorted(ckpts)[-NUM_GRAD_EPOCHS:]
internal_node_count = tree.w_list.shape[0]

for curr_path_node in PATH_NODE_IDS:
    path_node_mask = [np.ones(width, dtype=bool) for width in HIDDEN_DIMS]
    path_node_mask[PATH_LAYER_IDX][:] = False
    path_node_mask[PATH_LAYER_IDX][curr_path_node] = True

    true_grad_dict = {}
    grad_alt_dict = {}
    W0_dict = {}
    for epoch in tqdm(analysis_epochs):
        state = ckpts[epoch]
        checkpoint_model = checkpoint_model_from_state(
            seed_result["out"]["model"],
            state,
        )
        x_eval = seed_result["data"]["x_eval"]
        labels = seed_result["data"]["y_eval"]
        outputs = checkpoint_model(
            torch.tensor(x_eval, dtype=torch.float32)
        ).detach().cpu().numpy()
        residues = np.exp(-labels * outputs) / (1 + np.exp(-labels * outputs))
        residues = (residues + 1000.0) / 2000.0

        acts = activation_tensor_by_layer_neuron_data(
            model=checkpoint_model,
            x_eval=x_eval,
        )
        total_lens = first_layer_neuron_weighted_lens(
            state_dict=state,
            activation_tensor=acts,
            first_layer_neuron_id=FIRST_LAYER_NEURON_ID,
            node_mask=path_node_mask,
        )
        total_lens *= residues * -labels
        grad_alt = total_lens @ x_eval / len(residues)

        grad_check = log_loss_gradients(
            model=checkpoint_model,
            x=x_eval,
            y=labels,
        )
        true_grad = grad_check["hidden_layers.0.weight"][
            FIRST_LAYER_NEURON_ID
        ].numpy()
        true_grad_dict[epoch] = true_grad
        grad_alt_dict[epoch] = grad_alt
        W0_dict[epoch] = state["hidden_layers.0.weight"][
            FIRST_LAYER_NEURON_ID
        ].numpy()

    true_grad_comps = {}
    alt_grad_comps = {}
    true_grad_comps_numpy = np.zeros(
        (len(true_grad_dict), internal_node_count)
    )
    alt_grad_comps_numpy = np.zeros(
        (len(grad_alt_dict), internal_node_count)
    )
    for row_idx, epoch in enumerate(analysis_epochs):
        true_grad_comps[epoch] = odt_wts.numpy() @ true_grad_dict[epoch]
        true_grad_comps_numpy[row_idx] = true_grad_comps[epoch]
        alt_grad_comps[epoch] = odt_wts.numpy() @ grad_alt_dict[epoch]
        alt_grad_comps_numpy[row_idx] = alt_grad_comps[epoch]

    for odt_node_idx in range(internal_node_count - 1):
        plt.plot(analysis_epochs, alt_grad_comps_numpy[:, odt_node_idx])
    plt.axhline(0.0, color="gray", linewidth=1)
    plt.plot(
        analysis_epochs,
        alt_grad_comps_numpy[:, -1],
        "r--",
    )
    plt.title(
        f"Through node {curr_path_node} in hidden layer {PATH_LAYER_IDX + 1}"
    )
    plt.show()


In [ ]:
from tqdm.std import tqdm

FIRST_LAYER_NEURON_ID = min(30, HIDDEN_DIMS[0] - 1)
PATH_LAYER_IDX = 1 if len(HIDDEN_DIMS) > 1 else 0
SELECTED_PATH_NODE_IDS = (
    [min(34, HIDDEN_DIMS[PATH_LAYER_IDX] - 1)]
    if PATH_LAYER_IDX > 0
    else [FIRST_LAYER_NEURON_ID]
)
NUM_GRAD_EPOCHS = 100
analysis_epochs = sorted(ckpts)[-NUM_GRAD_EPOCHS:]
internal_node_count = tree.w_list.shape[0]

path_node_mask = [np.ones(width, dtype=bool) for width in HIDDEN_DIMS]
path_node_mask[PATH_LAYER_IDX][:] = False
path_node_mask[PATH_LAYER_IDX][SELECTED_PATH_NODE_IDS] = True

true_grad_dict = {}
grad_alt_dict = {}
W0_dict = {}
for epoch in tqdm(analysis_epochs):
    state = ckpts[epoch]
    checkpoint_model = checkpoint_model_from_state(
        seed_result["out"]["model"],
        state,
    )
    x_eval = seed_result["data"]["x_eval"]
    labels = seed_result["data"]["y_eval"]
    outputs = checkpoint_model(
        torch.tensor(x_eval, dtype=torch.float32)
    ).detach().cpu().numpy()
    residues = np.exp(-labels * outputs) / (1 + np.exp(-labels * outputs))
    residues = (residues + 1000.0) / 2000.0

    acts = activation_tensor_by_layer_neuron_data(
        model=checkpoint_model,
        x_eval=x_eval,
    )
    total_lens = first_layer_neuron_weighted_lens(
        state_dict=state,
        activation_tensor=acts,
        first_layer_neuron_id=FIRST_LAYER_NEURON_ID,
        node_mask=path_node_mask,
    )
    total_lens *= residues * -labels
    grad_alt = total_lens @ x_eval / len(residues)

    grad_check = log_loss_gradients(
        model=checkpoint_model,
        x=x_eval,
        y=labels,
    )
    true_grad = grad_check["hidden_layers.0.weight"][
        FIRST_LAYER_NEURON_ID
    ].numpy()
    true_grad_dict[epoch] = true_grad
    grad_alt_dict[epoch] = grad_alt
    W0_dict[epoch] = state["hidden_layers.0.weight"][
        FIRST_LAYER_NEURON_ID
    ].numpy()

true_grad_comps = {}
alt_grad_comps = {}
true_grad_comps_numpy = np.zeros(
    (len(true_grad_dict), internal_node_count)
)
alt_grad_comps_numpy = np.zeros(
    (len(grad_alt_dict), internal_node_count)
)
for row_idx, epoch in enumerate(analysis_epochs):
    true_grad_comps[epoch] = odt_wts.numpy() @ true_grad_dict[epoch]
    true_grad_comps_numpy[row_idx] = true_grad_comps[epoch]
    alt_grad_comps[epoch] = odt_wts.numpy() @ grad_alt_dict[epoch]
    alt_grad_comps_numpy[row_idx] = alt_grad_comps[epoch]


In [ ]:
PLOT_ODT_NODE_IDS = [
    node_idx
    for node_idx in [0, 2, 5]
    if node_idx < internal_node_count
]
for odt_node_idx in PLOT_ODT_NODE_IDS:
    plt.plot(
        analysis_epochs,
        alt_grad_comps_numpy[:, odt_node_idx],
    )
plt.axhline(0.0, color="gray", linewidth=1)
plt.plot(
    analysis_epochs,
    alt_grad_comps_numpy[:, -1],
    "r--",
)
plt.show()


In [ ]:
comparison_epoch = analysis_epochs[0]
true_grad_dict[comparison_epoch] / grad_alt_dict[comparison_epoch]

In [ ]:
gradient_epochs = sorted(grad_alt_dict)
internal_node_count = tree.w_list.shape[0]
true_grad_comps = {}
alt_grad_comps = {}
true_grad_comps_numpy = np.zeros(
    (len(gradient_epochs), internal_node_count)
)
alt_grad_comps_numpy = np.zeros(
    (len(gradient_epochs), internal_node_count)
)
for row_idx, epoch in enumerate(gradient_epochs):
    true_grad_comps[epoch] = odt_wts.numpy() @ true_grad_dict[epoch]
    true_grad_comps_numpy[row_idx] = true_grad_comps[epoch]
    alt_grad_comps[epoch] = odt_wts.numpy() @ grad_alt_dict[epoch]
    alt_grad_comps_numpy[row_idx] = alt_grad_comps[epoch]


In [ ]:
# All selected paths through FIRST_LAYER_NEURON_ID.
for odt_node_idx in range(internal_node_count - 1):
    plt.plot(
        gradient_epochs,
        alt_grad_comps_numpy[:, odt_node_idx],
    )
plt.axhline(0.0, color="gray", linewidth=1)
plt.plot(
    gradient_epochs,
    alt_grad_comps_numpy[:, -1],
    "r--",
)
plt.show()

In [ ]:
plt.plot(true_grad_comps.keys(),true_grad_comps_numpy[:,0] )
plt.plot(alt_grad_comps.keys(),alt_grad_comps_numpy[:,0] )
plt.plot([min(true_grad_comps.keys()),max(true_grad_comps.keys())],[0,0])

In [ ]:
grad_epochs = sorted(grad_comps)
PLOT_GRAD_ODT_NODE_IDS = [
    node_idx
    for node_idx in [0, 2, 6, 14, internal_node_count - 1]
    if 0 <= node_idx < internal_node_count
]
for odt_node_idx in dict.fromkeys(PLOT_GRAD_ODT_NODE_IDS):
    plt.plot(
        grad_epochs,
        grad_comps_numpy[:, odt_node_idx],
        label=f"ODT node {odt_node_idx}",
    )
plt.axhline(0.0, color="gray", linewidth=1)
plt.legend()

In [ ]:
CUMULATIVE_ODT_NODE_IDS = [
    node_idx
    for node_idx in [0, 1, 2, internal_node_count - 1]
    if 0 <= node_idx < internal_node_count
]
for odt_node_idx in dict.fromkeys(CUMULATIVE_ODT_NODE_IDS):
    plt.plot(
        np.cumsum(grad_comps_numpy[:, odt_node_idx]),
        label=f"ODT node {odt_node_idx}",
    )
plt.axhline(0.0, color="gray", linewidth=1)
plt.legend()

In [ ]:
W0_epochs = sorted(W0_dict)
internal_node_count = tree.w_list.shape[0]
W0_comps = {}
W0_comps_numpy = np.zeros((len(W0_epochs), internal_node_count))
for row_idx, epoch in enumerate(W0_epochs):
    W0_comps[epoch] = odt_wts.numpy() @ W0_dict[epoch]
    W0_comps_numpy[row_idx] = W0_comps[epoch]


In [ ]:
PLOT_ODT_NODE_IDS = [
    node_idx
    for node_idx in [0, 2, 6, 14, 30, 24]
    if node_idx < internal_node_count
]
for odt_node_idx in PLOT_ODT_NODE_IDS:
    plt.plot(
        W0_epochs,
        W0_comps_numpy[:, odt_node_idx],
        label=f"ODT node {odt_node_idx}",
    )
plt.axhline(0.0, color="gray", linewidth=1)
plt.legend()

In [ ]:
print(np.max(np.abs(odt_wts.numpy()@grad_alt)))
print(np.argmax(np.abs(odt_wts.numpy()@grad_alt)))
print(odt_wts.numpy()@grad_alt)
print(odt_wts.numpy()@true_grad)

In [ ]:
ODT_NODE_ID = min(30, tree.w_list.shape[0] - 1)
first_layer_weight = HIDDEN_WEIGHTS[0][FIRST_LAYER_NEURON_ID]
(
    first_layer_weight @ odt_wts[ODT_NODE_ID]
    / (
        torch.norm(first_layer_weight)
        * torch.norm(odt_wts[ODT_NODE_ID])
    )
)

In [ ]:
u_ODT_node = seed_result["tree"].w_list[ODT_NODE_ID]
grad_alt_cosine = grad_alt @ u_ODT_node / (
    np.linalg.norm(u_ODT_node) * np.linalg.norm(grad_alt)
)
print(
    "Cosine similarity between grad_alt for first-layer neuron",
    FIRST_LAYER_NEURON_ID,
    "and ODT node",
    ODT_NODE_ID,
    "at epoch",
    EPOCH,
    ":",
    grad_alt_cosine,
)
print("Norm of grad_alt:", np.linalg.norm(grad_alt))

In [ ]:
from layer_lenses.relu_analysis import summarize_neuron_leaf_activity

checkpoint_model = checkpoint_model_from_state(seed_result["out"]["model"], ckpts[EPOCH])

neuron_leaf_activity_summary = summarize_neuron_leaf_activity(
    model=checkpoint_model,
    x_eval=seed_result["data"]["x_eval"],
    tree=seed_result["tree"],
)

# display(neuron_leaf_activity_summary)
# show(
#     neuron_leaf_activity_summary,
#     scrollY="500px",
#     scrollX=True,
#     search=True,
#     paging=True,
#     pageLength=25,
# )

In [ ]:
# Build: neuron -> activity vector over ODT leaves
# Input: neuron_leaf_activity_summary (DataFrame from earlier cell)
# Output: neuron_to_leaf_activity (dict)

from layer_lenses.relu_analysis import neuron_leaf_activity_dict

neuron_to_leaf_activity, neuron_activity_leaf_order = neuron_leaf_activity_dict(
    neuron_leaf_activity_summary,
    key_mode="layer_local",
)

print(f"Built dictionary for {len(neuron_to_leaf_activity)} neurons.")
print(f"Vector length per neuron: {len(neuron_activity_leaf_order)} leaves")
print("Example key:", next(iter(neuron_to_leaf_activity)))
print("Example vector:", neuron_to_leaf_activity[next(iter(neuron_to_leaf_activity))])

In [ ]:
EXAMPLE_ACTIVITY_LAYER_IDX = min(1, len(HIDDEN_DIMS) - 1)
EXAMPLE_ACTIVITY_NEURON_ID = min(
    7,
    HIDDEN_DIMS[EXAMPLE_ACTIVITY_LAYER_IDX] - 1,
)
neuron_to_leaf_activity[
    (EXAMPLE_ACTIVITY_LAYER_IDX, EXAMPLE_ACTIVITY_NEURON_ID)
]

## Start/end metrics

This block computes start/end metrics for the selected seed.

In [ ]:
if seed_result is None:
    raise ValueError("Run the training cell first to populate seed_result.")

rows = collect_start_end_metrics(seed_result)
metrics_table = pd.DataFrame(rows).sort_values(["timepoint"]).reset_index(drop=True)
display(metrics_table)

## Quick visualization

Bar plots for start vs end test error and train log loss for the selected seed.

In [ ]:
if seed_result is None:
    raise ValueError("Run the training cell first to populate seed_result.")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(metrics_table["timepoint"], metrics_table["test_error"])
axes[0].set_title("Test error")

axes[1].bar(metrics_table["timepoint"], metrics_table["train_log_loss"])
axes[1].set_title("Train log loss")

plt.tight_layout()
plt.show()